In [ ]:
import re, os, joblib
from google.colab import drive


In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/BAD_L3Cube(Random_forest)_models"
os.makedirs(save_dir, exist_ok=True)

Mounted at /content/drive


In [ ]:
!pip install -q scikit-learn openpyxl transformers torch

import pandas as pd
import re
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score
from scipy.sparse import hstack
import os
import joblib
import gc
import time
from sklearn.model_selection import train_test_split

# =====================================================
# SPEED OPTIMIZATIONS
# =====================================================

# Force CPU usage if GPU is slow/unavailable
torch.backends.cudnn.benchmark = True
torch.set_num_threads(4)

# =====================================================
# Paths for BAD dataset
# =====================================================
datasets_a = {
    "Train": "/content/BAD_train-binary.xlsx",
    "Validation": "/content/BAD_val-binary.xlsx",
    "Test": "/content/BAD_test-binary.xlsx"
}
datasets_b = {
    "Train": "/content/BAD_train-multi.xlsx",
    "Validation": "/content/BAD_val-multi.xlsx",
    "Test": "/content/BAD_test-multi.xlsx"
}
datasets_c = {
    "Train": "/content/BAD_train-multi.xlsx",
    "Validation": "/content/BAD_val-multi.xlsx",
    "Test": "/content/BAD_test-multi.xlsx"
}

# =====================================================
# ULTRA-FAST Text Cleaning
# =====================================================
class FastTextCleaner:
    def __init__(self):
        # Pre-compile regex for speed
        self.clean_pattern = re.compile(r"http\S+|@\w+|#\w+|\d+")
        self.char_pattern = re.compile(r"[^\w\s\u0980-\u09FF]")
        self.space_pattern = re.compile(r"\s+")

    def clean_bangla_text(self, text):
        text = str(text)
        text = self.clean_pattern.sub("", text)
        text = self.char_pattern.sub("", text)
        return self.space_pattern.sub(" ", text).strip()

# Global cleaner instance
cleaner = FastTextCleaner()

# =====================================================
# ULTRA-FAST L3Cube Feature Extractor
# =====================================================
class FastL3CubeFeatureExtractor:
    def __init__(self, model_name="l3cube-pune/mahahate-bert"):
        print("Loading BERT model...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()

        # Use GPU only if available and faster
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        self.model.half()  # Use FP16 for speed

        print(f"Model loaded on {self.device}")

    def extract_features(self, texts, max_length=64, batch_size=32):  # Reduced max_length and increased batch_size
        """Extract BERT features with aggressive speed optimizations"""

        print(f"Extracting BERT features for {len(texts)} texts...")

        # Limit dataset size for speed
        if len(texts) > 8000:
            print(f"Sampling {8000} texts from {len(texts)} for speed...")
            indices = np.random.choice(len(texts), 8000, replace=False)
            texts = [texts[i] for i in sorted(indices)]

        features = []

        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                if i % (batch_size * 10) == 0:
                    print(f"Processing batch {i//batch_size + 1}/{(len(texts) + batch_size - 1)//batch_size}")

                batch_texts = texts[i:i+batch_size]

                # Truncate texts for speed
                batch_texts = [str(text)[:200] for text in batch_texts]  # Limit text length

                inputs = self.tokenizer(
                    batch_texts,
                    max_length=max_length,
                    padding='max_length',
                    truncation=True,
                    return_tensors='pt'
                )

                inputs = {k: v.to(self.device) for k, v in inputs.items()}

                outputs = self.model(**inputs)
                cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                features.extend(cls_embeddings)

                # Clear GPU memory
                del inputs, outputs
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        result = np.array(features, dtype=np.float32)  # Use float32 for memory
        print(f"BERT features shape: {result.shape}")
        return result

# =====================================================
# ULTRA-FAST TF-IDF Vectorizers (REDUCED FEATURES)
# =====================================================
print("Initializing TF-IDF vectorizers...")
word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    max_features=1500,  # Reduced from 3000
    max_df=0.95,       # Skip very common words
    min_df=3           # Skip very rare words
)
char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 4),  # Reduced from (3,5)
    max_features=1500,   # Reduced from 3000
    max_df=0.95,
    min_df=3
)
combined_vectorizer = FeatureUnion([('word', word_vectorizer), ('char', char_vectorizer)])

# =====================================================
# ULTRA-FAST Training Function
# =====================================================
def ultra_fast_run_subtask(dataset_paths, subtask_name):
    print(f"\n ULTRA-FAST {subtask_name} 🚀")
    start_time = time.time()

    # Load data
    print("Loading datasets...")
    train_df = pd.read_excel(dataset_paths["Train"])
    val_df = pd.read_excel(dataset_paths["Validation"])
    test_df = pd.read_excel(dataset_paths["Test"])
    train_df = pd.concat([train_df, val_df], ignore_index=True)

    # Drop NaN rows
    train_df = train_df.dropna(subset=['cleaned', 'Label'])
    test_df = test_df.dropna(subset=['cleaned', 'Label'])

    print(f"Data loaded: {len(train_df)} train, {len(test_df)} test")

    # SPEED HACK: Sample large datasets
    if len(train_df) > 12000:
        print(f"Sampling training data from {len(train_df)} to 10000 for speed...")
        train_df = train_df.sample(n=10000, random_state=42).reset_index(drop=True)

    # Fast text cleaning
    print("Fast text cleaning...")
    train_df['clean_text'] = [cleaner.clean_bangla_text(text) for text in train_df['cleaned']]
    test_df['clean_text'] = [cleaner.clean_bangla_text(text) for text in test_df['cleaned']]

    # Label encoding
    label_enc = LabelEncoder()
    y_train = label_enc.fit_transform(train_df['Label'])
    y_test = label_enc.transform(test_df['Label'])

    print(f"Classes: {list(label_enc.classes_)}")

    # ULTRA-FAST TF-IDF features
    print("Extracting TF-IDF features...")
    tfidf_start = time.time()
    X_train_tfidf = combined_vectorizer.fit_transform(train_df['clean_text'])
    X_test_tfidf = combined_vectorizer.transform(test_df['clean_text'])
    print(f"TF-IDF completed in {time.time() - tfidf_start:.2f}s")
    print(f"TF-IDF shape: {X_train_tfidf.shape}")

    # ULTRA-FAST BERT features
    print("Extracting BERT features...")
    bert_start = time.time()
    bert_extractor = FastL3CubeFeatureExtractor()
    X_train_bert = bert_extractor.extract_features(train_df['clean_text'].tolist())
    X_test_bert = bert_extractor.extract_features(test_df['clean_text'].tolist())

    # Handle sampling mismatch - FIXED: Use .shape[0] instead of len() for sparse matrices
    if len(X_train_bert) != X_train_tfidf.shape[0]:
        # Take corresponding TF-IDF features
        X_train_tfidf = X_train_tfidf[:len(X_train_bert)]
        y_train = y_train[:len(X_train_bert)]
        print(f"Adjusted training size to {len(X_train_bert)}")

    print(f"BERT completed in {time.time() - bert_start:.2f}s")

    # Clear memory
    del bert_extractor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Combine features
    print("Combining TF-IDF + BERT features...")
    X_train_combined = hstack([X_train_tfidf, X_train_bert])
    X_test_combined = hstack([X_test_tfidf, X_test_bert])
    print(f"Combined features shape: {X_train_combined.shape}")

    # Clear intermediate variables
    del X_train_tfidf, X_test_tfidf, X_train_bert, X_test_bert
    gc.collect()

    # ULTRA-FAST Grid Search (reduced parameters)
    print("Fast hyperparameter tuning...")
    param_grid = {
        'n_estimators': [100, 200],           # Reduced from [100, 300]
        'max_depth': [20, None],              # Reduced from [10, 30, None]
        'min_samples_split': [2, 5],          # Same
        'min_samples_leaf': [1],              # Reduced from [1, 2]
        'max_features': ['sqrt']              # Reduced from ['sqrt', 'log2']
    }

    grid_rf = GridSearchCV(
        estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
        param_grid=param_grid,
        scoring='f1_macro',
        cv=2,  # Reduced from 3
        n_jobs=1,  # Avoid nested parallelism
        verbose=1
    )

    grid_search_start = time.time()
    grid_rf.fit(X_train_combined, y_train)
    model = grid_rf.best_estimator_
    print(f"Grid search completed in {time.time() - grid_search_start:.2f}s")

    print(f"\nBest Parameters: {grid_rf.best_params_}")
    print(f"Best CV Score: {grid_rf.best_score_:.4f}")

    # Predictions
    print("Making predictions...")
    preds = model.predict(X_test_combined)

    # Results
    print("\nClassification Report:")
    print(classification_report(y_test, preds, target_names=[str(cls) for cls in label_enc.classes_]))

    macro_f1 = f1_score(y_test, preds, average='macro')
    acc = accuracy_score(y_test, preds)
    print(f"Accuracy: {acc:.4f}, Macro F1: {macro_f1:.4f}")


    # Save model, encoder, and vectorizers
    try:
        save_path = '/content/drive/MyDrive/BAD_L3Cube(Random_forest)_models'
        os.makedirs(save_path, exist_ok=True)

        # Save RF model
        joblib.dump(model, f"{save_path}/fast_rf_{subtask_name}.joblib")

        # Save label encoder
        joblib.dump(label_enc, f"{save_path}/fast_label_encoder_{subtask_name}.joblib")

        # Save TF-IDF vectorizers
        joblib.dump(word_vectorizer, f"{save_path}/fast_word_vectorizer_{subtask_name}.joblib")
        joblib.dump(char_vectorizer, f"{save_path}/fast_char_vectorizer_{subtask_name}.joblib")
        joblib.dump(combined_vectorizer, f"{save_path}/fast_combined_vectorizer_{subtask_name}.joblib")

        print(f"✅ Model, encoder, and vectorizers saved for {subtask_name}")

    except Exception as e:
        print(f"⚠️ Could not save for {subtask_name}: {str(e)}")


# =====================================================
# RUN ALL SUBTASKS WITH SPEED MONITORING
# =====================================================
print("ULTRA-FAST BERT+RF EXECUTION STARTED ")
print("Target: Complete all subtasks in under 30 minutes")

total_start_time = time.time()

try:
    results = {}
    results['subtask_a'] = ultra_fast_run_subtask(datasets_a, "subtask_a")
    results['subtask_b'] = ultra_fast_run_subtask(datasets_b, "subtask_b")
    results['subtask_c'] = ultra_fast_run_subtask(datasets_c, "subtask_c")

    total_time = time.time() - total_start_time

    print(f"\n FINAL RESULTS ")
    print(f"Total execution time: {total_time/60:.2f} minutes")
    print("-" * 50)
    for subtask, score in results.items():
        print(f"{subtask.upper()}: F1 = {score:.4f}")
    print("-" * 50)
    avg_f1 = np.mean(list(results.values()))
    print(f"Average F1-Score: {avg_f1:.4f}")
    print(f" SUCCESS: Completed in {total_time/60:.2f} minutes! ")

except Exception as e:
    print(f" Error occurred: {str(e)}")
    import traceback
    traceback.print_exc()

Initializing TF-IDF vectorizers...
ULTRA-FAST BERT+RF EXECUTION STARTED 
Target: Complete all subtasks in under 30 minutes

 ULTRA-FAST subtask_a 🚀
Loading datasets...
Data loaded: 12742 train, 1416 test
Sampling training data from 12742 to 10000 for speed...
Fast text cleaning...
Classes: [np.int64(0), np.int64(1)]
Extracting TF-IDF features...
TF-IDF completed in 8.72s
TF-IDF shape: (10000, 3000)
Extracting BERT features...
Loading BERT model...
Model loaded on cuda
Extracting BERT features for 10000 texts...
Sampling 8000 texts from 10000 for speed...
Processing batch 1/250
Processing batch 11/250
Processing batch 21/250
Processing batch 31/250
Processing batch 41/250
Processing batch 51/250
Processing batch 61/250
Processing batch 71/250
Processing batch 81/250
Processing batch 91/250
Processing batch 101/250
Processing batch 111/250
Processing batch 121/250
Processing batch 131/250
Processing batch 141/250
Processing batch 151/250
Processing batch 161/250
Processing batch 171/250


Traceback (most recent call last):
  File "/tmp/ipython-input-576910466.py", line 309, in <cell line: 0>
    print(f"{subtask.upper()}: F1 = {score:.4f}")
                                    ^^^^^^^^^^^
TypeError: unsupported format string passed to NoneType.__format__


In [ ]:
# =====================================================
# Install dependencies
# =====================================================
# !pip install transformers datasets torch scikit-learn openpyxl joblib -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
import numpy as np
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
import re, os, joblib
from google.colab import drive

# =====================================================
# Mount Google Drive
# =====================================================
drive.mount("/content/drive", force_remount=True)
save_path = "/content/drive/MyDrive/BAD_models"
os.makedirs(save_path, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================
# Data Cleaning
# =====================================================
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|@\w+|#\w+|\d+|[^\w\s\u0980-\u09FF]", "", text)  # keep Bangla chars
    return re.sub(r"\s+", " ", text).strip()

# =====================================================
# Dataset Class
# =====================================================
class BADDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}
        enc.pop("token_type_ids", None)
        enc["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return enc

# =====================================================
# Multi-Level Attention
# =====================================================
class MultiLevelAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn1 = nn.Linear(hidden_size, 1)
        self.attn2 = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states):
        w1 = torch.softmax(self.attn1(hidden_states), dim=1)
        context1 = torch.sum(w1 * hidden_states, dim=1)

        w2 = torch.softmax(self.attn2(hidden_states), dim=1)
        context2 = torch.sum(w2 * hidden_states, dim=1)

        return context1 + context2

# =====================================================
# Transformer Model
# =====================================================
class TransformerWithPCA(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.attn = MultiLevelAttention(self.bert.config.hidden_size)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        attn_output = self.attn(hidden_states)
        pooled = self.dropout(attn_output)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
        return {"loss": loss, "logits": logits}

# =====================================================
# Training & Evaluation
# =====================================================
def train_and_eval(dataset_paths, subtask):
    print(f"\n===== Running {subtask} =====")

    # Load & merge train + validation
    train_df = pd.read_excel(dataset_paths["Train"])
    val_df = pd.read_excel(dataset_paths["Validation"])
    test_df = pd.read_excel(dataset_paths["Test"])
    train_df = pd.concat([train_df, val_df], ignore_index=True)

    # Prepare clean text & labels
    train_df = train_df.dropna(subset=["cleaned", "Label"]).copy()
    test_df = test_df.dropna(subset=["cleaned", "Label"]).copy()

    train_df["clean"] = train_df["cleaned"].apply(clean_text)
    test_df["clean"] = test_df["cleaned"].apply(clean_text)

    le = LabelEncoder()
    y_train = le.fit_transform(train_df["Label"])
    y_test = le.transform(test_df["Label"])

    tokenizer = AutoTokenizer.from_pretrained("l3cube-pune/mahahate-bert")

    train_dataset = BADDataset(train_df["clean"].tolist(), y_train, tokenizer)
    test_dataset = BADDataset(test_df["clean"].tolist(), y_test, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64)

    model = TransformerWithPCA("l3cube-pune/mahahate-bert", num_labels=len(le.classes_)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    # Train
    for epoch in range(5):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, total=len(train_loader)):
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = out["loss"]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

    # Evaluate
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in test_loader:
            labels = batch["labels"].to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            pred = torch.argmax(out["logits"], dim=1)
            preds.extend(pred.cpu().numpy())
            trues.extend(labels.cpu().numpy())

    print(classification_report(trues, preds, target_names=[str(c) for c in le.classes_]))

    macro_f1 = f1_score(trues, preds, average="macro")
    print(f"Macro F1: {macro_f1:.4f}")

    # Save everything
    model_dir = f"{save_path}/{subtask}"
    os.makedirs(model_dir, exist_ok=True)
    torch.save(model.state_dict(), f"{model_dir}/model.pt")
    tokenizer.save_pretrained(model_dir)
    joblib.dump(le, f"{model_dir}/label_encoder.pkl")
    print(f"✅ Saved {subtask} to {model_dir}")

    return model, tokenizer, le

# =====================================================
# Run for BAD Dataset (A, B, C)
# =====================================================
datasets_a = {
    "Train": "/content/BAD_train-binary.xlsx",
    "Validation": "/content/BAD_val-binary.xlsx",
    "Test": "/content/BAD_test-binary.xlsx"
}
datasets_b = {
    "Train": "/content/BAD_train-multi.xlsx",
    "Validation": "/content/BAD_val-multi.xlsx",
    "Test": "/content/BAD_test-multi.xlsx"
}
datasets_c = datasets_b  # same as multi

model_a, tokenizer_a, le_a = train_and_eval(datasets_a, "subtask_a")
model_b, tokenizer_b, le_b = train_and_eval(datasets_b, "subtask_b")
model_c, tokenizer_c, le_c = train_and_eval(datasets_c, "subtask_c")

# =====================================================
# Reload Example (later, without retraining)
# =====================================================
def load_model(subtask, num_labels):
    model_dir = f"{save_path}/{subtask}"
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    le = joblib.load(f"{model_dir}/label_encoder.pkl")
    model = TransformerWithPCA("l3cube-pune/mahahate-bert", num_labels=num_labels)
    model.load_state_dict(torch.load(f"{model_dir}/model.pt", map_location=device))
    model.to(device)
    model.eval()
    print(f"🔄 Reloaded {subtask} from {model_dir}")
    return model, tokenizer, le


Mounted at /content/drive

===== Running subtask_a =====


tokenizer_config.json:   0%|          | 0.00/305 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

100%|██████████| 200/200 [04:59<00:00,  1.50s/it]


Epoch 2 Loss: 0.2305


100%|██████████| 200/200 [05:00<00:00,  1.50s/it]


Epoch 3 Loss: 0.1703


 58%|█████▊    | 116/200 [02:54<02:06,  1.51s/it]

In [ ]:
from google.colab import drive
import os

drive.mount('/content/gdrive')  # different mount point

save_path = "/content/gdrive/MyDrive/bad_models"
os.makedirs(save_path, exist_ok=True)

print(f"✅ Drive mounted. Models will be stored in: {save_path}")


Mounted at /content/gdrive
✅ Drive mounted. Models will be stored in: /content/gdrive/MyDrive/bad_models


In [ ]:
import joblib

# Save Subtask A
joblib.dump(model_a, f"{save_path}/fast_rf_subtask_a.joblib")
joblib.dump(le_a, f"{save_path}/fast_label_encoder_subtask_a.joblib")

# Save Subtask B
joblib.dump(model_b, f"{save_path}/fast_rf_subtask_b.joblib")
joblib.dump(le_b, f"{save_path}/fast_label_encoder_subtask_b.joblib")

# Save Subtask C
joblib.dump(model_c, f"{save_path}/fast_rf_subtask_c.joblib")
joblib.dump(le_c, f"{save_path}/fast_label_encoder_subtask_c.joblib")

print("✅ All models and encoders saved into Google Drive")


NameError: name 'model_a' is not defined